# Employee Data — Gold Layer Aggregation
### Silver → Analytics-Ready Summaries & Denormalized Views

**Purpose:** This notebook reads the cleansed silver-layer dimensions and facts, then builds **four gold tables** optimized for reporting and downstream business-case analysis. Each gold table is a wide, denormalized view designed to answer specific analytical questions without requiring joins at query time.

**Tables Produced:**

| Gold Table | Description | Key Derived Fields |
|------------|-------------|--------------------|
| `gold_employee_summary` | One row per employee with compensation, tenure, and role attributes | `total_compensation`, `seniority_level`, `promotion_eligible` |
| `gold_district_compensation` | District-level headcounts, avg pay, and promotion pipeline | `avg_total_compensation`, `promotion_eligible_pct` |
| `gold_seniority_analysis` | Seniority-band breakdown with comp ranges and degree diversity | `min/max_compensation`, `distinct_degree_types` |
| `gold_certification_overview` | Teacher cert records with active/expired status and duration | `cert_status`, `cert_duration_days` |

**Input Tables (Silver Layer):**
- `dim_employee`, `dim_district`, `dim_certification`, `dim_pab_item`, `dim_date_employee`
- `fact_pay_and_benefits`, `fact_pab_lineitem`, `fact_teacher_certification`

---
_Upstream: [employee_silver](#) · Downstream: [employee_business_cases](#)_

In [0]:
# ── PySpark imports ──
# Standard toolkit for column operations, type casting, and timestamp generation.
# Used throughout the notebook for joins, aggregations, and derived columns.

from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType, StringType, BooleanType, TimestampType

In [0]:
# ── Medallion architecture paths (ADLS Gen2) ──
# Gold layer reads from silver/ and writes analytics-ready output to gold/.

root_path = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net/"
bronze_path = f"{root_path}/bronze"
silver_path = f"{root_path}/silver"
gold_path = f"{root_path}/gold"

# ── Unity Catalog schema references ──
bronze_sch = "bronze_employee"
silver_sch = "silver_employee"
gold_sch = "gold_employee"
employee_db = "employeedatacatalog"

In [0]:
# ============================================================
# Create gold schema and load all silver tables into DataFrames
# ============================================================
# The gold schema holds denormalized, analytics-ready tables.
# We load all silver dimensions and facts here so they're
# available for the gold table builds below.

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {employee_db}.{gold_sch}")

# ── Dimensions (cleansed reference data) ──
dim_employee    = spark.table(f"{employee_db}.{silver_sch}.dim_employee")      # master employee record
dim_district    = spark.table(f"{employee_db}.{silver_sch}.dim_district")      # school district lookup
dim_cert        = spark.table(f"{employee_db}.{silver_sch}.dim_certification") # certification codes
dim_pab_item    = spark.table(f"{employee_db}.{silver_sch}.dim_pab_item")      # pay item descriptions
dim_date        = spark.table(f"{employee_db}.{silver_sch}.dim_date_employee") # date dimension

# ── Facts (transactional/event data) ──
fact_pab        = spark.table(f"{employee_db}.{silver_sch}.fact_pay_and_benefits")       # annual comp totals
fact_lineitem   = spark.table(f"{employee_db}.{silver_sch}.fact_pab_lineitem")           # granular line items
fact_cert       = spark.table(f"{employee_db}.{silver_sch}.fact_teacher_certification")  # cert events

print("Silver tables loaded | Gold schema ready")

In [0]:
# ============================================================
# GOLD TABLE: gold_employee_summary
# ============================================================
# The master employee record for all downstream analytics.
# Joins dim_employee + dim_district + fact_pay_and_benefits into
# a single wide row per employee with:
#   - Demographics: name, zip, hire date, degree, role flags
#   - District context: district name
#   - Compensation: reg pay, OT, other, benefits, total
#   - HR attributes: tenure, seniority, promotion eligibility

gold_emp_summary = dim_employee.alias("e") \
    .join(dim_district.alias("d"),                              # Add district name
          col("e.DISTRICT_ID") == col("d.DISTRICT_ID"), "left") \
    .join(
        fact_pab.alias("p"),                                    # Add compensation data
        col("e.EMP_ID") == col("p.EMP_ID"),
        "left"
    ) \
    .select(
        col("e.EMP_ID"),
        col("e.EMPLOYEE_NAME"),
        col("e.DISTRICT_ID"),
        col("d.DISTRICT_NAME"),
        col("e.ZIPCODE"),
        col("e.HIREDATE"),
        col("e.HIGHEST_EARNED_DEGREE"),
        col("e.IS_ADMIN"),
        col("e.IS_TEACHER"),
        col("e.EDU_EMAIL"),
        col("e.PREVIOUS_EXPERIENCE_YEARS"),
        col("e.years_of_service"),                              # Derived in silver
        col("e.total_experience"),                              # Derived in silver
        col("e.seniority_level"),                               # Derived in silver
        col("e.promotion_eligible"),                            # Derived in silver
        col("p.TAX_YEAR"),
        col("p.REG_PAY"),
        col("p.OVERTIME_PAY"),
        col("p.OTHER_PAY"),
        col("p.TOTAL_BENEFITS"),
        col("p.total_compensation"),                            # Derived in silver
        col("p.DATE_LAST_CALC"),
    ) \
    .withColumn("_gold_ingested_at", current_timestamp())       # Audit timestamp


gold_emp_summary.limit(10).display()                            # Preview only 10 rows to conserve resources

# Persist to gold Delta and register in UC
gold_emp_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_employee_summary")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.gold_employee_summary
          USING DELTA
          LOCATION '{gold_path}/gold_employee_summary'""")

In [0]:
# ============================================================
# GOLD TABLE: gold_district_compensation
# ============================================================
# Rolls up employee-level data to district-level summaries.
# Provides headcounts, avg pay metrics, and promotion pipeline
# stats for each school district — ideal for executive dashboards.

gold_district_comp = gold_emp_summary \
    .groupBy("DISTRICT_ID", "DISTRICT_NAME") \
    .agg(
        count("EMP_ID").alias("employee_count"),                           # Total headcount
        sum(when(col("IS_TEACHER") == True, 1).otherwise(0)).alias("teacher_count"),
        sum(when(col("IS_ADMIN") == True, 1).otherwise(0)).alias("admin_count"),
        round(avg("years_of_service"), 1).alias("avg_years_of_service"),
        round(avg("total_experience"), 1).alias("avg_total_experience"),
        round(avg("total_compensation"), 2).alias("avg_total_compensation"),
        round(sum("total_compensation"), 2).alias("total_compensation_budget"),  # District-wide spend
        round(avg("REG_PAY"), 2).alias("avg_reg_pay"),
        round(avg("OVERTIME_PAY"), 2).alias("avg_overtime_pay"),
        round(avg("TOTAL_BENEFITS"), 2).alias("avg_benefits"),
        sum(when(col("promotion_eligible") == True, 1).otherwise(0)).alias("promotion_eligible_count"),
        round(
            sum(when(col("promotion_eligible") == True, 1).otherwise(0)) / count("EMP_ID") * 100, 1
        ).alias("promotion_eligible_pct")                                 # % of workforce ready for promotion
    ) \
    .withColumn("_gold_ingested_at", current_timestamp()) \
    .orderBy("DISTRICT_ID")


gold_district_comp.limit(10).display()                                    # Preview only 10 rows to conserve resources

# Persist to gold Delta and register in UC
gold_district_comp.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_district_compensation")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.gold_district_compensation
          USING DELTA
          LOCATION '{gold_path}/gold_district_compensation'""")

In [0]:
# ============================================================
# GOLD TABLE: gold_seniority_analysis
# ============================================================
# Groups employees by seniority level (Entry → Senior) and computes
# compensation ranges, role mix (teacher/admin), and promotion
# readiness. Sorted in logical progression from Entry to Senior.

gold_seniority = gold_emp_summary \
    .groupBy("seniority_level") \
    .agg(
        count("EMP_ID").alias("employee_count"),                           # Band headcount
        round(avg("years_of_service"), 1).alias("avg_years_of_service"),
        round(avg("total_experience"), 1).alias("avg_total_experience"),
        round(avg("total_compensation"), 2).alias("avg_total_compensation"),
        round(min("total_compensation"), 2).alias("min_compensation"),     # Comp floor in this band
        round(max("total_compensation"), 2).alias("max_compensation"),     # Comp ceiling in this band
        sum(when(col("promotion_eligible") == True, 1).otherwise(0)).alias("promotion_eligible_count"),
        round(
            sum(when(col("promotion_eligible") == True, 1).otherwise(0)) / count("EMP_ID") * 100, 1
        ).alias("promotion_eligible_pct"),
        countDistinct("HIGHEST_EARNED_DEGREE").alias("distinct_degree_types"),  # Degree diversity
        sum(when(col("IS_TEACHER") == True, 1).otherwise(0)).alias("teacher_count"),
        sum(when(col("IS_ADMIN") == True, 1).otherwise(0)).alias("admin_count")
    ) \
    .withColumn("_gold_ingested_at", current_timestamp()) \
    .orderBy(
        # Custom sort: logical seniority progression Entry → Senior
        when(col("seniority_level") == "Entry (<2 yrs)", 1)
        .when(col("seniority_level") == "Junior (2-4 yrs)", 2)
        .when(col("seniority_level") == "Mid-Level (5-9 yrs)", 3)
        .when(col("seniority_level") == "Mid-Senior (10-19 yrs)", 4)
        .when(col("seniority_level") == "Senior (20+ yrs)", 5)
    )


gold_seniority.limit(10).display()                                        # Preview only 10 rows to conserve resources

# Persist to gold Delta and register in UC
gold_seniority.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_seniority_analysis")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.gold_seniority_analysis
          USING DELTA
          LOCATION '{gold_path}/gold_seniority_analysis'""")

In [0]:
# ============================================================
# GOLD TABLE: gold_certification_overview
# ============================================================
# Joins the teacher certification fact table with cert descriptions,
# employee names, and district info to build a complete certification
# record. Adds a human-readable cert_status label:
#   Active  → cert hasn't expired yet
#   Expired → cert is past its expiry date
#   Unknown → no expiry date on record

gold_cert = fact_cert.alias("fc") \
    .join(dim_cert.alias("c"),                                  # Add certification descriptions
          col("fc.CERT_ID") == col("c.CERT_ID"), "left") \
    .join(
        dim_employee.select("EMP_ID", "EMPLOYEE_NAME", "DISTRICT_ID").alias("e"),
        col("fc.T_EMP_ID") == col("e.EMP_ID"),                  # Link to employee
        "left"
    ) \
    .join(dim_district.alias("d"),                              # Add district name
          col("e.DISTRICT_ID") == col("d.DISTRICT_ID"), "left") \
    .select(
        col("fc.T_EMP_ID").alias("EMP_ID"),
        col("e.EMPLOYEE_NAME"),
        col("d.DISTRICT_NAME"),
        col("c.CERT_ID"),
        col("c.STATE_CERT_CODE"),
        col("c.CERT_DESC"),
        col("fc.DATE_EFFECTIVE"),
        col("fc.DATE_EXPIRES"),
        col("fc.is_active"),
        col("fc.cert_duration_days"),
        when(col("fc.is_active") == True, lit("Active"))         # Human-readable status label
            .when(col("fc.DATE_EXPIRES").isNull(), lit("Unknown"))
            .otherwise(lit("Expired")).alias("cert_status")
    ) \
    .withColumn("_gold_ingested_at", current_timestamp())       # Audit timestamp


gold_cert.limit(10).display()                                   # Preview only 10 rows to conserve resources

# Persist to gold Delta and register in UC
gold_cert.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_certification_overview")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {employee_db}.{gold_sch}.gold_certification_overview
          USING DELTA
          LOCATION '{gold_path}/gold_certification_overview'""")